# tensor-item-scalar — worked example 3: .item() vs .tolist() — safe extraction with numel check

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-item-scalar`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

`.item()` only works when the tensor has exactly one element (`numel() == 1`). Calling it on a multi-element tensor raises `RuntimeError`. For multi-element tensors, `.tolist()` converts the entire tensor to a (nested) Python list. A numel check lets you write a single function that handles both cases safely.

## Worked solution

**Step 1 — When to use `.item()`.**
If `x.numel() == 1`, any shape works: `(1,)`, `(1, 1)`, `()`. All yield a scalar via `.item()`.

**Step 2 — When to use `.tolist()`.**
If `x.numel() > 1`, call `.tolist()`. A 1-D tensor gives a flat list; a 2-D tensor gives a list of lists.

**Step 3 — Guard pattern.**
`if x.numel() == 1: return x.item()` else `return x.tolist()`. Do not blindly call `.item()` without checking.

**Step 4 — Practical use.**
This pattern appears in metric loggers that accept both scalar tensors (loss, accuracy) and multi-element tensors (class probabilities) and need to produce JSON-serialisable output.

In [ ]:
import torch as t

def safe_extract(x: t.Tensor):
    """Return a Python scalar for single-element tensors, a list otherwise."""
    if x.numel() == 1:
        return x.item()     # scalar: float, int, or bool
    return x.tolist()       # multi-element: nested Python list

# Demonstrate different shapes
t.manual_seed(0)
print(safe_extract(t.tensor(3.5)))           # scalar, returns float 3.5
print(safe_extract(t.tensor([[7.0]])))       # (1,1) single element -> float
print(safe_extract(t.tensor([1, 2, 3])))    # 3 elements -> list
print(safe_extract(t.randn(2, 3)))           # (2,3) -> list of lists
print(type(safe_extract(t.tensor(5.0))))    # <class 'float'>
print(type(safe_extract(t.randn(4))))       # <class 'list'>